# Task 2 — Notebook 2: Create the Labels

## Notebook Objective

* Build the target label (`is_late`) by comparing the actual delivery date with the estimated delivery date.
* Validate the label by checking it on a few real orders before using it.
* Analyze the class distribution to determine whether there is a class imbalance and measure its size.
* Build one final **artifact**: the labeled table.




In [1]:
import pandas as pd

ml_table = pd.read_parquet("artifacts/ml_table.parquet")
print(ml_table.shape)

(99441, 20)


### 1 Check Order Status and Missing Delivery Dates

* Check the number of orders for each `order_status`.
* Check how many orders have a missing `order_delivered_customer_date`.
* The actual delivery date is needed to build the `is_late` label.
* Orders without an actual delivery date cannot be labeled as late or on-time.


In [2]:
print(ml_table["order_status"].value_counts())
print("\nMissing order_delivered_customer_date:", ml_table["order_delivered_customer_date"].isna().sum())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Missing order_delivered_customer_date: 2965


### 2 Check Delivered Orders with Missing Delivery Date

* Find orders marked as `delivered` but with a missing `order_delivered_customer_date`.
* Count how many such orders exist.
* Display important columns to inspect these orders.
* This check helps ensure the data is reliable before creating the `is_late` label.


In [3]:
delivered_but_missing_date = ml_table[
    (ml_table["order_status"] == "delivered") &
    (ml_table["order_delivered_customer_date"].isna())
]

print("Delivered orders with missing delivery date:", delivered_but_missing_date.shape[0])
delivered_but_missing_date[[
    "order_id", "order_status", "order_purchase_timestamp",
    "order_delivered_customer_date", "order_estimated_delivery_date"
]]

Delivered orders with missing delivery date: 8


,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,NaT,2018-07-19



* We found **8 orders** with `order_status = "delivered"` but missing `order_delivered_customer_date`. Since there is no reliable source to recover the missing date, these orders will be **removed from the labeled dataset** rather than guessing the delivery date.

* Orders that are **not delivered** (`canceled`, `shipped`, `processing`, `unavailable`, etc.) do not have a valid delivery outcome. They cannot be classified as either **late** or **on-time**.

* Therefore, the labeled dataset will include only orders where:

  * `order_status == "delivered"`
  * `order_delivered_customer_date` is not missing.

This ensures that the `is_late` label represents **delivery delay only**, without mixing it with cancellations or other order outcomes.


In [4]:
labeled_table = ml_table[
    (ml_table["order_status"] == "delivered") &
    (ml_table["order_delivered_customer_date"].notna())
].copy()

print("Original ml_table:", ml_table.shape)
print("Labeled table (delivered only, valid date):", labeled_table.shape)
print("Dropped rows:", ml_table.shape[0] - labeled_table.shape[0])

Original ml_table: (99441, 20)
Labeled table (delivered only, valid date): (96470, 20)
Dropped rows: 2971


### 3. Build the Label

The `is_late` label is created by comparing the actual delivery date with the estimated delivery date:

* `is_late = 1` → The order was delivered **after** the estimated delivery date.
* `is_late = 0` → The order was delivered **on time or earlier**.

This label will be the **target variable** for the ML model.


In [5]:
labeled_table["is_late"] = (
    labeled_table["order_delivered_customer_date"] > labeled_table["order_estimated_delivery_date"]
).astype(int)

labeled_table[[
    "order_id", "order_delivered_customer_date",
    "order_estimated_delivery_date", "is_late"
]].head(10)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-10 21:25:13,2017-10-18,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-07 15:27:45,2018-08-13,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-17 18:06:29,2018-09-04,0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-02 00:28:42,2017-12-15,0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-16 18:17:02,2018-02-26,0
5,a4591c265e18cb1dcee52889e2d8acc3,2017-07-26 10:57:55,2017-08-01,0
7,6514b8ad8028c9f2cc2374ded245783f,2017-05-26 12:55:51,2017-06-07,0
8,76c6e866289321a7c93b82b54852dc33,2017-02-02 14:08:10,2017-03-06,0
9,e69bfb5eb88e0ed6a785585b27e16dbf,2017-08-16 17:14:30,2017-08-23,0
10,e6ce16cb79ec1d90b1da9085a6118aeb,2017-05-29 11:18:31,2017-06-07,0


## 4. Validate the Label on Real Orders

Before using the label, we manually check a few random orders to make sure that the actual delivery date is correctly compared with the estimated delivery date and that the resulting `is_late` value is correct.


In [6]:
sample_check = labeled_table.sample(5, random_state=42)[[
    "order_id", "order_delivered_customer_date",
    "order_estimated_delivery_date", "is_late"
]]
sample_check["days_diff"] = (
    sample_check["order_delivered_customer_date"] - sample_check["order_estimated_delivery_date"]
).dt.days

sample_check

,order_id,order_delivered_customer_date,order_estimated_delivery_date,is_late,days_diff
9504,c6a73b421eb3e92ce86dbfbbd3530a8f,2018-03-13 17:13:17,2018-03-19,0,-6
31123,b132124ca9d69faf63989e08a5851151,2018-06-04 19:26:52,2018-06-08,0,-4
27721,7e25a1c58e68fa94300358caad65b944,2017-09-08 17:21:38,2017-09-25,0,-17
74022,8418eb39cd68b52032566797b8f1bd11,2017-12-12 21:13:48,2017-12-20,0,-8
36187,fe1ec86f91f3b5b6bc46fc3e4b8262cd,2017-12-13 01:16:52,2017-12-15,0,-2


### Manual Validation

For each sampled order, verify that:

* `days_diff > 0` corresponds to `is_late = 1`.
* `days_diff <= 0` corresponds to `is_late = 0`.

If any mismatch is found, the label creation logic should be reviewed before proceeding.


## 5. Class Distribution and Imbalance Check

We calculate the number and percentage of orders in each class (`is_late = 0` and `is_late = 1`) to understand the class distribution and check whether the dataset has a class imbalance.


In [8]:
class_counts = labeled_table["is_late"].value_counts()
class_pct = labeled_table["is_late"].value_counts(normalize=True) * 100

print("Counts:\n", class_counts)
print("\nPercentages:\n", class_pct.round(2))

Counts:
 is_late
0    88644
1     7826
Name: count, dtype: int64

Percentages:
 is_late
0    91.89
1     8.11
Name: proportion, dtype: float64


### Class Distribution Results

* **On-time orders (`is_late = 0`): 88,644 orders (91.89%)**
* **Late orders (`is_late = 1`): 7,826 orders (8.11%)**
* The dataset has a **noticeable class imbalance**, since the minority class represents only **8.11%** of the data.
* Therefore, **Accuracy alone is not sufficient** for model evaluation. Metrics such as **F1-score, Precision, Recall, and PR-AUC** should also be considered.


### 6.Final Sanity Checks

Before saving the labeled table, we verify that:

* Each `order_id` is unique.
* There are no missing values in `is_late`.
* The `is_late` label contains only `0` or `1`.
* If all checks pass, the labeled table is ready for the next stage.


In [9]:
assert labeled_table["order_id"].duplicated().sum() == 0, "Duplicate order_id in labeled table!"
assert labeled_table["is_late"].isna().sum() == 0, "Missing label values found!"
assert set(labeled_table["is_late"].unique()) <= {0, 1}, "Unexpected label values!"

print("All sanity checks passed.")
print("Final labeled_table shape:", labeled_table.shape)

All sanity checks passed.
Final labeled_table shape: (96470, 21)


### 7 Save the Artifact

Save the `labeled_table` as the final artifact so that Notebook 3 can load it directly for the Train/Validation/Test split without repeating the previous preprocessing and labeling steps.


In [10]:
import os

os.makedirs("artifacts", exist_ok=True)
labeled_table.to_parquet("artifacts/labeled_table.parquet", index=False)
print("Saved artifacts/labeled_table.parquet ->", labeled_table.shape)

Saved artifacts/labeled_table.parquet -> (96470, 21)
